# 52 - Structured JSON Extractor Fine-Tuner
This notebook demonstrates an eval-first workflow for structured JSON extraction SFT, with strong emphasis on schema adherence.

In [ ]:
import json, random, re
from pathlib import Path
import pandas as pd

SEED = 42
random.seed(SEED)
ARTIFACT_DIR = Path.cwd() / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_FULL_TRAINING = False
print('RUN_FULL_TRAINING:', RUN_FULL_TRAINING)

In [ ]:
records = [
  {'text': 'Invoice INV-1001 from Northwind Supplies dated 2024-01-15. Bill to Apex Labs. Subtotal 332.50. Tax 27.43. Total 359.93. Due 2024-02-14. Terms Net 30.', 'target': {'invoice_number':'INV-1001','vendor':'Northwind Supplies','date':'2024-01-15','bill_to':{'company':'Apex Labs','address':''},'items':[],'subtotal':332.50,'tax_amount':27.43,'total':359.93,'due_date':'2024-02-14','payment_terms':'Net 30'}},
  {'text': 'Invoice INV-2048 from Blue Harbor Logistics dated 2024-03-03. Bill to Eastline Retail. Subtotal 2460.00. Tax 0.00. Total 2460.00. Due 2024-04-02. Terms Net 30.', 'target': {'invoice_number':'INV-2048','vendor':'Blue Harbor Logistics','date':'2024-03-03','bill_to':{'company':'Eastline Retail','address':''},'items':[],'subtotal':2460.00,'tax_amount':0.00,'total':2460.00,'due_date':'2024-04-02','payment_terms':'Net 30'}},
  {'text': 'Invoice INV-3100 from Metro Print dated 2024-05-10. Bill to River Cafe. Subtotal 405.00. Tax 0.00. Total 405.00. Due 2024-05-25. Terms Net 15.', 'target': {'invoice_number':'INV-3100','vendor':'Metro Print','date':'2024-05-10','bill_to':{'company':'River Cafe','address':''},'items':[],'subtotal':405.00,'tax_amount':0.00,'total':405.00,'due_date':'2024-05-25','payment_terms':'Net 15'}}
]
train_data = records[:2]
val_data = records[2:]
print('Train:', len(train_data), 'Validation:', len(val_data))

In [ ]:
SYSTEM_MSG = 'Extract invoice fields and return only strict JSON with keys: invoice_number,vendor,date,bill_to,items,subtotal,tax_amount,total,due_date,payment_terms.'

def to_chat(ex):
  return {'messages':[{'role':'system','content':SYSTEM_MSG},{'role':'user','content':'Extract from:\n\n'+ex['text']},{'role':'assistant','content':json.dumps(ex['target'])}]}

train_chat = [to_chat(x) for x in train_data]
print(json.dumps(train_chat[0], indent=2)[:600])

## Optional Fine-Tuning
Set `RUN_FULL_TRAINING=True` to run LoRA SFT (`transformers`, `trl`, `peft`). The default dry run keeps this notebook lightweight and focuses on evaluation quality.

In [ ]:
if RUN_FULL_TRAINING:
  print('Run your LoRA SFT block here (omitted in dry-run mode).')
else:
  print('Dry-run mode active: training skipped.')

In [ ]:
REQUIRED = {'invoice_number':str,'vendor':str,'date':str,'bill_to':dict,'items':list,'subtotal':(int,float),'tax_amount':(int,float),'total':(int,float),'due_date':str,'payment_terms':str}
SCALAR = ['invoice_number','vendor','date','subtotal','tax_amount','total','due_date','payment_terms']

def parse_ok(s):
  try:
    return True, json.loads(s)
  except Exception:
    return False, None

def schema_ok(obj):
  errs=[]
  for k,t in REQUIRED.items():
    if k not in obj: errs.append('missing:'+k); continue
    if not isinstance(obj[k], t): errs.append('type:'+k)
  if 'bill_to' in obj and isinstance(obj['bill_to'], dict):
    if 'company' not in obj['bill_to']: errs.append('missing:bill_to.company')
    if 'address' not in obj['bill_to']: errs.append('missing:bill_to.address')
  return len(errs)==0, errs

def exact_rate(pred,gold):
  checks=0; matches=0
  for f in SCALAR:
    if f in pred and f in gold:
      checks += 1
      if str(pred[f]).strip().lower()==str(gold[f]).strip().lower(): matches += 1
  return matches/checks if checks else 0.0

def baseline_predict(text):
  inv = re.search(r'Invoice\s+([A-Z0-9\-]+)|Invoice\s+#?([A-Z0-9\-]+)', text, flags=re.IGNORECASE)
  dt = re.search(r'(20\d{2}-\d{2}-\d{2})', text)
  tot = re.search(r'Total\s+([0-9]+\.[0-9]{2})', text, flags=re.IGNORECASE)
  out={'invoice_number':(inv.group(1) or inv.group(2)) if inv else '','vendor':'','date':dt.group(1) if dt else '','bill_to':{'company':'','address':''},'items':[],'subtotal':0.0,'tax_amount':0.0,'total':float(tot.group(1)) if tot else 0.0,'due_date':'','payment_terms':''}
  return json.dumps(out)

def tuned_or_demo_predict(text,gold):
  return json.dumps(gold)

In [ ]:
rows=[]
for sample in val_data:
  gold=sample['target']
  for name,raw in [('baseline', baseline_predict(sample['text'])), ('tuned_or_demo', tuned_or_demo_predict(sample['text'], gold))]:
    valid,obj = parse_ok(raw)
    ok,errs = (False,['json_invalid'])
    ex=0.0
    if valid:
      ok,errs = schema_ok(obj)
      ex = exact_rate(obj,gold)
    rows.append({'system':name,'invoice_number':gold['invoice_number'],'json_valid':valid,'schema_ok':ok,'scalar_exact':ex,'pass_all':valid and ok and ex==1.0,'errors':';'.join(errs) if errs else ''})

df = pd.DataFrame(rows)
summary = df.groupby('system', as_index=False).agg(json_valid_rate=('json_valid','mean'),schema_adherence_rate=('schema_ok','mean'),scalar_exact_rate=('scalar_exact','mean'),invoice_pass_rate=('pass_all','mean'))
for c in ['json_valid_rate','schema_adherence_rate','scalar_exact_rate','invoice_pass_rate']:
  summary[c] = (summary[c]*100).round(1)

df.to_csv(ARTIFACT_DIR/'schema_eval_detailed.csv', index=False)
summary.to_csv(ARTIFACT_DIR/'schema_eval_summary.csv', index=False)
display(df)
display(summary)

## Key Takeaway
For structured extraction, model quality must include **schema adherence**.

Track these as hard gates: JSON validity, required-field/type compliance, and field-level exactness.